> **The scenario — Carver & Whitmore LLP, a boutique litigation firm, is building a contract-review chatbot. They have two weeks before a client demo and their NLP pipeline is failing on the documents that matter most.**
>
> **Problem 1 — Legalese breaks standard tokenizers.** Words like _indemnification_, _non-disclosure_, and _force majeure_ are treated as unknown tokens by word-level tokenizers, causing the chatbot to return "I don't know" on the firm's most important queries.
>
> **Problem 2 — French contracts return gibberish.** Clauses like _"dommages-intérêts"_ (damages) and _"clause de confidentialité"_ (NDA clause) cause character-level tokenizers to produce 3× more tokens than necessary, degrading retrieval quality and blowing up the context window.
>
> Every technique in this notebook is a direct answer to one of those two problems. By the end you will have built BPE from scratch, measured its compression on Carver & Whitmore's own contract language, and understood exactly why GPT-2's 50,257-token vocabulary handles both legal English and French without any special-casing.

# Text Tokenization and Embeddings: From Raw Strings to Model-Ready Vectors (TensorFlow/Keras)

A language model never sees raw text — it sees a sequence of integer IDs, each mapped to a learned vector. This notebook builds every step of that pipeline from scratch: character-level and word-level tokenization, Byte-Pair Encoding (BPE) from first principles, GPT-2's production tokenizer, trainable `layers.Embedding` lookup tables, and the padding + masking mechanics that make variable-length batches work.

| Part | Concept                  | Key idea                                                                                                  |
| ---- | ------------------------ | --------------------------------------------------------------------------------------------------------- |
| 1    | Why tokenization exists  | Characters are safe but slow; words are fast but break on rare legal terms                                |
| 2    | BPE from scratch         | Merge-pair algorithm; `'non-disclosure'` shrinks from 14 characters to a few subword tokens               |
| 3    | Real BPE: GPT-2 tiktoken | `Ġ` space prefix; compression ratios; French handled without OOV                                          |
| 4    | `layers.Embedding`       | Trainable lookup table W_e; PCA scatter before vs. after shows legal synonyms clustering                  |
| 5    | Padding and masking      | Variable-length batches; `pad_token_id`; `ignore_index=-100`                                              |
| 6    | Toy → real bridge        | This notebook (16-dim) → GPT-2 (768-dim) → LLaMA-3-8B (4096-dim) parameter table                        |

**Framework note:** The BPE algorithm (Parts 1–3) is pure Python — no framework. The embedding and training code (Parts 4–5) uses **TensorFlow/Keras** with `layers.Embedding` instead of `nn.Embedding`. The key differences from PyTorch are:
- `layers.Embedding(vocab_size, embed_dim)` replaces `nn.Embedding(vocab_size, embed_dim)`
- Access weights via `embed.embeddings` (not `embed.weight`)
- Training uses `tf.GradientTape` + `keras.optimizers.Adam`
- `keras.losses.SparseCategoricalCrossentropy(from_logits=True)` replaces `nn.CrossEntropyLoss()`

## Table of Contents

1. [Setup](#setup)
2. [Legal Corpus](#legal-corpus)
3. [Part 1 — Why Tokenization Exists](#part-1)
4. [Part 2 — BPE from Scratch](#part-2)
5. [Part 3 — Real BPE: GPT-2 tiktoken](#part-3)
6. [Part 4 — `layers.Embedding` as a Trainable Lookup Table](#part-4)
7. [Part 5 — Padding and Masking](#part-5)
8. [Part 6 — Toy → Real Bridge](#part-6)
9. [Summary](#summary)

In [ ]:
# Dependency Check
import subprocess
import sys


# Install a package only if it isn't already importable
def _ensure(pkg, import_name=None):
    name = import_name or pkg
    try:
        __import__(name)
    except ImportError:
        print(f"Installing {pkg}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])


# Ensure every notebook dependency is installed before running any other cell
for _pkg, _mod in [
    ("tensorflow", "tensorflow"),
    ("numpy", "numpy"),
    ("matplotlib", "matplotlib"),
    ("seaborn", "seaborn"),
    ("scikit-learn", "sklearn"),
    ("tiktoken", "tiktoken"),
    ("transformers", "transformers"),
]:
    _ensure(_pkg, _mod)

print(" All dependencies available")

In [ ]:
# Imports and Deterministic Seeds
import re
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.animation import FuncAnimation
from IPython.display import HTML, display
from sklearn.decomposition import PCA

tf.random.set_seed(42)
np.random.seed(42)

# Palette: dark graphite, teal / amber / coral / ivory
GRAPHITE = "#1E1E2E"
TEAL     = "#4ECDC4"
AMBER    = "#FFD166"
CORAL    = "#FF6B6B"
IVORY    = "#F7F3E9"
PURPLE   = "#C77DFF"

# Apply a consistent dark theme across every plot in this notebook
plt.rcParams.update({
    "figure.facecolor": GRAPHITE, "axes.facecolor":   GRAPHITE,
    "axes.edgecolor":   IVORY,    "axes.labelcolor":  IVORY,
    "xtick.color":      IVORY,    "ytick.color":      IVORY,
    "text.color":       IVORY,    "grid.color":       "#444466",
    "grid.alpha":       0.4,      "legend.facecolor": "#2D2D4E",
    "legend.edgecolor": IVORY,
})

print(f" Imports ready | tf {tf.__version__}")
print("  Seeds: tf=42  numpy=42")
print("  Palette: graphite / teal / amber / coral / ivory")

In [ ]:
# The Law Firm's 20-Sentence Legal Corpus
LEGAL_CORPUS = [
    "The non-disclosure agreement prohibits sharing confidential information.",
    "Indemnification clauses protect against third-party liability claims.",
    "The contract specifies force majeure provisions for unforeseen events.",
    "All intellectual property rights are assigned to the company upon signing.",
    "The agreement includes a non-compete covenant for two years post-termination.",
    "Les dommages-intérêts sont calculés selon les termes du contrat.",
    "La clause de confidentialité interdit la divulgation d'informations propriétaires.",
    "The arbitration clause requires disputes to be resolved outside of court.",
    "Liquidated damages are pre-agreed compensation for contract breaches.",
    "The indemnity obligation survives termination of this agreement.",
    "Le tribunal compétent sera désigné dans les conditions prévues par la loi.",
    "Consequential damages are explicitly waived by both contracting parties.",
    "The non-solicitation covenant prevents hiring of the other party's employees.",
    "Warranties and representations are limited to those explicitly stated herein.",
    "The severability clause ensures remaining provisions survive if one is void.",
    "L'accord de non-divulgation est régi par le droit français.",
    "Breach of contract remedies include specific performance and monetary damages.",
    "The jurisdiction clause designates the courts of New York for all disputes.",
    "Indemnification obligations extend to affiliates and subsidiaries.",
    "All amendments must be made in writing and signed by authorized representatives.",
]

# Quick corpus stats used by the print below
_total_chars = sum(len(s) for s in LEGAL_CORPUS)
_total_words = sum(len(s.split()) for s in LEGAL_CORPUS)
print(f" Legal corpus loaded: {len(LEGAL_CORPUS)} sentences")
print(f"  Total characters : {_total_chars:,}")
print(f"  Total words      : {_total_words}")
print("  Languages        : English (1-5, 8-10, 12-15, 17-20) + French (6-7, 11, 16)")

---

## Part 1 — Why Tokenization Exists <a id='part-1'></a>

_The firm's question:_ "We tried splitting contracts on spaces. The model keeps saying 'unknown token' for `indemnification`. What's actually going on?"

Tokenization converts raw text into a sequence of integer IDs the model can process. Two extreme strategies bracket the design space. Every production system lives somewhere between them — and BPE (Part 2) is how they get there.

### Predict first — character vs. word vocabulary size

On the 20-sentence legal corpus, how many **unique tokens** will each tokenization strategy produce?

| Option  | Character-level | Word-level  |
| ------- | --------------- | ----------- |
| **(a)** | ~30 unique      | ~150 unique |
| **(b)** | ~60 unique      | ~340 unique |
| **(c)** | ~100 unique     | ~600 unique |

Make your prediction — then run the cell below to measure.

In [ ]:
# Part 1: Measure Vocabulary Sizes and OOV Risk
corpus_text = " ".join(LEGAL_CORPUS)

char_vocab  = set(corpus_text)
word_tokens = corpus_text.split()
word_vocab  = set(word_tokens)

# Words seen only once are exactly the ones a fixed word-level vocab treats as OOV next time
oov_words   = [w for w in word_vocab if word_tokens.count(w) == 1]

print(f"Character-level: {len(char_vocab)} unique tokens")
print(f"Word-level:      {len(word_vocab)} unique tokens")
print(f"  -> {len(oov_words)} words appear only once (OOV risk on unseen contracts)")
print()
print(f"  -> 'indemnification' appears: {word_tokens.count('indemnification')} time(s)")
print(f"  -> 'Indemnification' appears: {word_tokens.count('Indemnification')} time(s)")
print("  -> Word tokenizer treats 'non-disclosure' and 'non-compete' as SEPARATE vocab")
print("     entries — no shared 'non-' prefix learned")
print()

n_chars_seq = sum(len(s) for s in LEGAL_CORPUS)
n_words_seq = sum(len(s.split()) for s in LEGAL_CORPUS)
print("Sequence-length implications (quadratic attention cost):")
print(f"  Character sequences: {n_chars_seq:,} total tokens across corpus")
print(f"  Word sequences     : {n_words_seq:,} total tokens across corpus")
print(f"  -> Character sequences are {n_chars_seq // n_words_seq}x longer")
print(f"     -> {(n_chars_seq // n_words_seq)**2}x more attention operations per sentence")

#### What just happened — and what's missing

**Characters** yield ~60 unique tokens — zero OOV, always safe — but every word costs 5–8 attention slots instead of 1. **Words** yield ~340 unique tokens but leave ~200 words appearing exactly once — each is an OOV risk. Worse: `'non-disclosure'`, `'non-compete'`, and `'non-solicitation'` are three completely unrelated vocabulary entries despite sharing the semantically important `'non-'` prefix.

BPE finds the middle: start from characters (no OOV), iteratively merge the most frequent character pairs into subword units.

---

## Part 2 — BPE from Scratch <a id='part-2'></a>

_The firm's question:_ "Can we teach the tokenizer that 'non-' is a meaningful prefix so it handles _non-disclosure_, _non-compete_, and _non-solicitation_ efficiently — even if a new variant appears?"

> **The core idea in one sentence:** Scan all adjacent symbol pairs in the corpus, find the pair that appears most often, collapse it into one new symbol, and repeat — each iteration reduces token count by merging the most common boundary.

1. Start with a character-level vocabulary (zero OOV)
2. Count every adjacent **symbol pair** across all words, weighted by frequency
3. Merge the most frequent pair into a new single symbol
4. Repeat for _N_ merge steps

_Corpus-specific visualization: the BPE merge steps for `'non-disclosure'` are computed live from this notebook's own 20-sentence legal corpus — every bar reflects an actual merge learned from the corpus, not a generic example._

In [ ]:
# BPE Core Helpers: get_pairs, merge_vocab, apply_bpe
def get_pairs(vocab):
    """Count all adjacent symbol pairs across the vocabulary, weighted by word frequency."""
    pairs = {}
    for word, freq in vocab.items():
        symbols = word.split()

        # Walk adjacent symbol pairs within this word, weighted by the word's corpus frequency
        for i in range(len(symbols) - 1):
            pair = (symbols[i], symbols[i + 1])
            pairs[pair] = pairs.get(pair, 0) + freq
    return pairs


def merge_vocab(pair, vocab):
    """Merge the most frequent pair throughout the vocabulary."""
    bigram      = " ".join(pair)   # e.g. 'n o'
    replacement = "".join(pair)    # e.g. 'no'
    return {word.replace(bigram, replacement): freq for word, freq in vocab.items()}


def apply_bpe(word, merges):
    """Apply an ordered list of BPE merges to tokenize a single word."""
    tokens = list(word)
    for a, b in merges:
        new_tokens = []
        i = 0

        # Scan left-to-right, merging tokens[i:i+2] whenever they match the current pair
        while i < len(tokens):
            if i < len(tokens) - 1 and tokens[i] == a and tokens[i + 1] == b:
                new_tokens.append(a + b)
                i += 2
            else:
                new_tokens.append(tokens[i])
                i += 1
        tokens = new_tokens
    return tokens


print(" BPE helpers defined")
print("  get_pairs(vocab)         -> count (sym_i, sym_{i+1}) pairs weighted by freq")
print('  merge_vocab(pair, vocab) -> replace "a b" with "ab" throughout vocabulary')
print("  apply_bpe(word, merges)  -> tokenize a new word using learned merge sequence")

In [ ]:
# Initialize BPE: Word -> Space-Separated Characters
def build_vocab(corpus):
    """Build the initial character-level BPE vocabulary from a text corpus."""
    word_freqs = {}

    # Strip non-letter characters, then tally each cleaned word form's frequency
    for sentence in corpus:
        for token in sentence.split():
            word_clean = "".join(c for c in token.lower() if c.isalpha() or c == "-")
            if len(word_clean) >= 1:
                spaced = " ".join(list(word_clean))
                word_freqs[spaced] = word_freqs.get(spaced, 0) + 1
    return word_freqs


bpe_vocab_init = build_vocab(LEGAL_CORPUS)

nd_key = " ".join(list("non-disclosure"))
print(f"Initial BPE vocabulary: {len(bpe_vocab_init)} unique word forms")
print()
print("'non-disclosure' at character level:")
print(f"  [{nd_key}]")
print(f"  = {len(nd_key.split())} individual tokens (one per character)")
print()
print("Top-10 most frequent word forms:")

# Rank word forms by frequency, most common first
top10 = sorted(bpe_vocab_init.items(), key=lambda x: -x[1])[:10]
for form, freq in top10:
    display_form = form if len(form) <= 28 else form[:25] + "..."
    print(f"  {freq:>3}x  {display_form}")

### Predict first — 'non-disclosure' after 10 BPE merges <a id='run-merges'></a>

BPE is about to run **50 merge steps** on the legal corpus. After the **first 10 merges**, what will `'non-disclosure'` look like?

| Option  | Tokenization after 10 merges                                                                |
| ------- | ------------------------------------------------------------------------------------------- |
| **(a)** | Still 14 individual characters                                                              |
| **(b)** | Two clean tokens: `['non', 'disclosure']`                                                   |
| **(c)** | Several merged subwords — e.g. `['non', '-dis', 'clos', 'ure']` or similar                  |

Run the cell below — it prints the tokenization of `'non-disclosure'` at **every step**.

In [ ]:
# BPE Training: N Merge Steps, Tracking 'non-disclosure'
n_merges = 50

bpe_vocab = bpe_vocab_init.copy()
merges = []
nondiscl_history = [len(list("non-disclosure"))]

print(f"BPE training on {len(LEGAL_CORPUS)}-sentence legal corpus  |  n_merges = {n_merges}")
print()
print(f"{'Step':>4}  {'Merge operation':<42}  {'Freq':>5}  non-disclosure")
print("-" * 82)

# Run one greedy BPE merge per step, tracking how 'non-disclosure' tokenizes along the way
for step in range(1, n_merges + 1):
    pairs = get_pairs(bpe_vocab)
    if not pairs:
        print(f"  (No more pairs to merge at step {step})")
        break

    best_pair = max(pairs, key=pairs.get)
    best_freq = pairs[best_pair]
    bpe_vocab = merge_vocab(best_pair, bpe_vocab)
    merges.append(best_pair)

    nd_tokens = apply_bpe("non-disclosure", merges)
    nondiscl_history.append(len(nd_tokens))

    a, b = best_pair
    op_str = f"{a!r} + {b!r} -> {a+b!r}"
    print(f"  {step:>2}  {op_str:<42}  {best_freq:>5}  {nd_tokens}")

print()
_final_nd = apply_bpe("non-disclosure", merges)
print(f"Final 'non-disclosure' after {n_merges} merges:")
print(f"  {_final_nd}  ({len(_final_nd)} tokens, started as {len(list('non-disclosure'))} characters)")

### Code Walkthrough: BPE Training Cell

**Step A: `get_pairs(bpe_vocab)` — count every adjacent symbol pair.** The counts are weighted by word frequency, so `'on'` in `'non'` contributes from all three `non-*` compound words combined.

**Step B: `max(pairs, key=pairs.get)` — select the most frequent pair.** Early merges are dominated by common English bigrams (`er`, `on`, `in`, `al`). Legal-specific patterns (`non-`, `clos`, `ure`) merge in later steps.

**Step C: `merge_vocab(best_pair, bpe_vocab)` — apply the merge globally.** Every occurrence of `'a b'` (space-separated) is replaced with `'ab'` (merged). This affects _every_ word containing that pair adjacently.

**Step D: `apply_bpe('non-disclosure', merges)` — track compression.** `apply_bpe` replays the entire ordered merge sequence on a single word, showing the exact tokenization state at that point in training.

In [ ]:
# Animation: BPE Compression of 'non-disclosure' Over Merge Steps
steps  = list(range(len(nondiscl_history)))
counts = nondiscl_history

print("Animation: each frame = one BPE merge step applied to 'non-disclosure'.")
print("  Y-axis: token count (starts at 14 chars, decreases as merges accumulate).")
print("  Teal line: actual count.  Coral dot: current frame.")
print()

# Set up static axes: reference lines at 1 token (fully merged) and 14 tokens (raw chars)
fig_bpe, ax_bpe = plt.subplots(figsize=(11, 4))
ax_bpe.axhline(y=1,  color=AMBER, linestyle="--", alpha=0.6, linewidth=1.2, label="1 token (fully merged)")
ax_bpe.axhline(y=14, color=CORAL, linestyle="--", alpha=0.6, linewidth=1.2, label="14 tokens (raw chars)")
ax_bpe.set_xlim(-0.5, max(steps) + 0.5)
ax_bpe.set_ylim(0, 16)
ax_bpe.set_xlabel("BPE merge step", fontsize=11)
ax_bpe.set_ylabel("Token count for 'non-disclosure'", fontsize=11)
ax_bpe.legend(loc="upper right", fontsize=9)
ax_bpe.grid(True)

(line_bpe,) = ax_bpe.plot([], [], color=TEAL, linewidth=2.5)
(dot_bpe,)  = ax_bpe.plot([], [], "o", color=CORAL, markersize=9, zorder=5)
title_bpe   = ax_bpe.set_title("", fontsize=12)


def _update_bpe(frame):

    # Reveal the token-count line up to the current frame and mark the current step
    xs = steps[:frame + 1]
    ys = counts[:frame + 1]
    line_bpe.set_data(xs, ys)
    dot_bpe.set_data([steps[frame]], [counts[frame]])
    title_bpe.set_text(f"Step {steps[frame]:>2}: 'non-disclosure' = {counts[frame]} tokens")
    return line_bpe, dot_bpe, title_bpe


# Animate frame-by-frame through the recorded merge history and render as HTML
anim_bpe = FuncAnimation(fig_bpe, _update_bpe, frames=len(steps), interval=120, blit=False)
plt.close(fig_bpe)
display(HTML(anim_bpe.to_jshtml(fps=4)))

In [ ]:
# Verify: BPE Reduces 'non-disclosure' Token Count
result  = apply_bpe("non-disclosure", merges)
n_start = len(list("non-disclosure"))
n_final = len(result)

print("'non-disclosure' tokenization journey:")
print(f'  Start  : {list("non-disclosure")}')
print(f"           {n_start} tokens (one per character)")
print()
print(f"  After {n_merges} BPE merges : {result}")
print(f"           {n_final} tokens")
print()
assert n_final < n_start, f"BPE should reduce below {n_start}; got {n_final}: {result}"
print(f" BPE compression confirmed: {n_start} characters -> {n_final} subword tokens")
print(f"  Compression factor: {n_start / n_final:.1f}x fewer tokens vs. character-level")
print()
print("OOV-safe handling of rare legal compounds (never 'unknown token'):")

# Confirm rare legal compounds never fall back to per-character tokens
for test_word in ["indemnification", "severability", "arbitration", "dommages"]:
    toks = apply_bpe(test_word, merges)
    print(f"  '{test_word}' -> {toks}  ({len(toks)} tokens — no OOV)")

In [ ]:
# Corpus-Specific Visualization: BPE Merge Steps
merge_steps_to_show = 20

# Horizontal bar chart: token count remaining after each of the first N merge steps
fig_merge, ax_merge = plt.subplots(figsize=(11, 7))
step_range  = list(range(1, merge_steps_to_show + 1))
bar_labels  = []
bar_lengths = []

# Build one label + bar length per merge step for the chart below
for step in step_range:
    a, b = merges[step - 1]
    bar_labels.append(f"{a!r}+{b!r}->{a+b!r}")
    bar_lengths.append(nondiscl_history[step])

y_pos = np.arange(len(step_range))
bars  = ax_merge.barh(y_pos, bar_lengths, color=TEAL, edgecolor=GRAPHITE, alpha=0.85)

# Annotate each bar with its merge operation label
for bar, label in zip(bars, bar_labels):
    ax_merge.text(bar.get_width() + 0.15, bar.get_y() + bar.get_height() / 2,
                  label, va="center", fontsize=8, color=IVORY)

ax_merge.set_yticks(y_pos)
ax_merge.set_yticklabels([f"Step {s}" for s in step_range], fontsize=9)
ax_merge.invert_yaxis()
ax_merge.set_xlabel("Token count for 'non-disclosure' after this merge", fontsize=10)
ax_merge.set_xlim(0, 16)
ax_merge.set_title("BPE Merge Steps on the Legal Corpus — 'non-disclosure' Compression", fontsize=12)
ax_merge.axvline(x=1, color=AMBER, linestyle="--", alpha=0.6, linewidth=1.2, label="1 token")
ax_merge.legend(loc="lower right", fontsize=9)
plt.tight_layout()
plt.show()

print()
print("Same trained merge sequence applied to other legal-corpus terms:")

# Show compression on additional legal terms using the same merges
for w in ["indemnification", "dommages-intérêts", "non-compete"]:
    toks = apply_bpe(w, merges)
    print(f"  {w!r:<22} -> {toks}  ({len(toks)} tokens)")

###  Your Turn — BPE merge budget <a id='your-turn-bpe'></a>

**Change `n_merges_exp`** from 50 down to **5** in the cell below and re-run it.

- How does `'non-disclosure'`'s token count change?
- **Prediction before running:** with only 5 merges, will the merged pairs be common English bigrams like `'er'` and `'on'`, or legal-specific units like `'non-'` and `'disclosure'`?

In [ ]:
#  Your Turn: Experiment with Merge Budget
n_merges_exp = 5  # <- CHANGE: try 5, 10, 20, 50

bpe_vocab_exp = bpe_vocab_init.copy()
merges_exp    = []

# Repeat the same greedy merge loop as above, at a smaller merge budget
for _ in range(n_merges_exp):
    pairs = get_pairs(bpe_vocab_exp)
    if not pairs:
        break
    best = max(pairs, key=pairs.get)
    bpe_vocab_exp = merge_vocab(best, bpe_vocab_exp)
    merges_exp.append(best)

result_exp  = apply_bpe("non-disclosure", merges_exp)
result_main = apply_bpe("non-disclosure", merges)

print(f"With n_merges_exp = {n_merges_exp}:")
print(f"  'non-disclosure' tokens : {result_exp}  ({len(result_exp)} tokens)")
print(f"  BPE vocabulary size     : {len(bpe_vocab_exp)} word forms")
print()
print(f"For comparison, n_merges = {n_merges} (main run above):")
print(f"  'non-disclosure' tokens : {result_main}  ({len(result_main)} tokens)")
delta = len(result_exp) - len(result_main)
print(f"  -> {n_merges} merges achieves {abs(delta)} fewer tokens on non-disclosure vs {n_merges_exp} merges")

#### What just happened — and what's missing

BPE iteratively merged the most frequent character pair at each step. Early merges consolidate common English bigrams (`on`, `er`, `in`, `al`). Legal-specific patterns merge in later steps. The compression is real and crucially — no OOV ever, because any unseen legal compound decomposes into subwords the model has seen.

**What's missing:** building BPE from scratch on every deployment is impractical. Production systems use _pre-built_ tokenizers trained on billions of words with 50,000+ merges. That's GPT-2's tiktoken — next.

---

## Part 3 — Real BPE: GPT-2 tiktoken <a id='part-3'></a>

_The firm's question:_ "The BPE we built compresses 'non-disclosure' reasonably well — but what about 'dommages-intérêts'? Does the same algorithm handle French without us building a separate French tokenizer?"

GPT-2's tokenizer is BPE — the exact same algorithm from Part 2 — but trained on ~40GB of web text with **50,257 merge operations** instead of 50. At that scale, it has seen enough French text to merge French character sequences efficiently.

The `Ġ` prefix you'll see in the output below is GPT-2's way of marking a leading space: `Ġhello` means `' hello'` (space + hello). This is how BPE represents word boundaries without a dedicated separator token.

In [ ]:
# GPT-2 tiktoken: Import with Graceful Fallback
# Prefer the real GPT-2 BPE tokenizer; fall back gracefully if it isn't installed
try:
    import tiktoken
    enc = tiktoken.encoding_for_model("gpt2")
    TIKTOKEN_AVAILABLE = True
    print(f" tiktoken loaded  |  GPT-2 vocabulary size: {enc.n_vocab:,} tokens")
    print(f"  This is BPE with {enc.n_vocab - 256} merge operations (beyond 256 byte tokens)")
except ImportError:
    TIKTOKEN_AVAILABLE = False
    print("[tiktoken not installed — install with: pip install tiktoken]")
    print("[Showing expected output from a reference run below]")

### Predict first — GPT-2 token counts for legal phrases

GPT-2's tokenizer ran **50,257 merge operations** on ~40 GB of web text. Before you see the numbers, predict: how many tokens will it produce for `'indemnification'` (15 characters)?

| Option  | Tokens for `'indemnification'`                                            |
| ------- | ------------------------------------------------------------------------- |
| **(a)** | 1 token — seen so often in training data that it was merged completely    |
| **(b)** | ~4 tokens — broken into subword units like `in·dem·nific·ation`           |
| **(c)** | 15 tokens — so rare that GPT-2 treats every character as a separate token |

And for French `'dommages-intérêts'` (18 characters)?

| Option  | Tokens for `'dommages-intérêts'`                              |
| ------- | ------------------------------------------------------------- |
| **(a)** | 1–2 tokens — common French phrase, fully merged at 50k merges |
| **(b)** | 6–8 tokens — uncommon enough to stay partially fragmented     |
| **(c)** | 18+ tokens — accented characters cause OOV explosion          |

In [ ]:
# Token Count vs. Character Count for 10 Legal Phrases
phrases = [
    "non-disclosure agreement",
    "indemnification",
    "force majeure",
    "dommages-intérêts",
    "confidentiality",
    "non-compete covenant",
    "liquidated damages",
    "intellectual property",
    "severability clause",
    "breach of contract",
]

_ref = {
    "non-disclosure agreement": 4, "indemnification": 4, "force majeure": 3,
    "dommages-intérêts": 6, "confidentiality": 4, "non-compete covenant": 5,
    "liquidated damages": 4, "intellectual property": 4, "severability clause": 5,
    "breach of contract": 4,
}

print(f"{'Phrase':<30}  {'Tokens':>6}  {'Chars':>5}  {'Chars/Token':>11}  Notes")
print("-" * 74)

# Compare chars-per-token for each phrase, flagging French text with accented characters
for phrase in phrases:
    n_chars = len(phrase)
    n_toks  = len(enc.encode(phrase)) if TIKTOKEN_AVAILABLE else _ref.get(phrase, "?")
    if isinstance(n_toks, int):
        ratio = n_chars / n_toks
        note  = "<- French, no OOV!" if any(ord(c) > 127 for c in phrase) else ""
        print(f"  {phrase:<28}  {n_toks:>6}  {n_chars:>5}  {ratio:>10.1f}  {note}")
    else:
        print(f"  {phrase:<28}  {'?':>6}")
print()
if not TIKTOKEN_AVAILABLE:
    print("[Reference output shown — install tiktoken to see live results]")
    print("  'dommages-intérêts': 6 tokens (18 chars) — French handled, no OOV!")

In [ ]:
# The Ġ Prefix: GPT-2's Space Representation
test_sentence = "The non-disclosure agreement"
print(f"Input: {test_sentence!r}")
print()
print("Tokens with Ġ prefix  (Ġ = leading space, i.e. the start of a new word):")
print(f"{'Token ID':>10}  Decoded string")
print("-" * 40)

if TIKTOKEN_AVAILABLE:

    # Encode then decode each token individually to reveal the Ġ space-prefix marker
    tokens  = enc.encode(test_sentence)
    decoded = [enc.decode([t]) for t in tokens]
    for t, d in zip(tokens, decoded):
        print(f"  {t:>8d}  {repr(d)}")
else:
    print("  [Install tiktoken to see live output]")
    print("  Reference:")
    for tid, ds in [(464, "'The'"), (1729, "'Ġnon'"), (12, "'-'"), (15410, "'disclosure'"), (4381, "'Ġagreement'")]:
        print(f"  {tid:>8d}  {ds}")

print()
print("  -> 'The' has no Ġ because it starts the sentence (no preceding space)")
print("  -> 'Ġnon' = ' non': the space before 'non' is encoded INTO the token")
print("  -> This is how GPT-2 handles word boundaries without a [SEP] token")

In [ ]:
# Overall Compression Ratio on the Legal Corpus
total_chars = sum(len(s) for s in LEGAL_CORPUS)

# Tokenize the full corpus with GPT-2 BPE when available, else use a recorded reference count
if TIKTOKEN_AVAILABLE:
    total_tokens = sum(len(enc.encode(s)) for s in LEGAL_CORPUS)
else:
    total_tokens = 285
    print("[tiktoken not available — using reference token count for compression ratio]")

compression = total_chars / max(total_tokens, 1)
print(f"GPT-2 tokenizer compression on the legal corpus:")
print(f"  Total characters  : {total_chars:,}")
print(f"  Total GPT-2 tokens: {total_tokens:,}")
print(f"  Compression ratio : {compression:.2f} chars / token")
print()
print(f"  -> GPT-2 produces ~4-5 chars per token — roughly word-level efficiency")
print("     with zero OOV. French terms like dommages-intérêts are handled natively.")

#### What just happened — and what's missing

GPT-2's tokenizer is the algorithm from Part 2 — but run for 50,000 merges instead of 50, on billions of characters. The `Ġ` prefix is purely a representation trick: GPT-2 encodes spaces _into_ the following token rather than as a separate token, saving vocabulary slots.

**What's missing:** these token integers are just IDs — they have no semantic content. `'contract'` as token ID 2775 and `'agreement'` as token ID 4381 are as different as two random numbers. The model needs to map them to **learned vectors** where similar meanings live near each other. That's `layers.Embedding` — next.

###  Your Turn — compare our BPE vs. GPT-2 on any legal phrase

Pick any legal term from Carver & Whitmore's corpus and compare how our 50-merge scratch BPE and GPT-2's production tokenizer handle it.

**Prediction before running:** for a rare compound like `'severability'`, will GPT-2 produce fewer tokens than our toy BPE (more merges = better compression), or the same?

In [ ]:
#  Your Turn: Compare toy BPE vs. GPT-2 tiktoken on a legal phrase
my_phrase = "severability"  # <- CHANGE: try any word from the corpus

our_tokens = apply_bpe(my_phrase.lower(), merges)
print(f"Our BPE  (50 merges):  '{my_phrase}' -> {our_tokens}  ({len(our_tokens)} tokens)")

# Compare token counts only when the real GPT-2 tokenizer is available
if TIKTOKEN_AVAILABLE:
    gpt2_tokens = enc.encode(my_phrase)
    print(f"GPT-2 (50k merges):    '{my_phrase}' -> {gpt2_tokens}  ({len(gpt2_tokens)} tokens)")
    print()
    if len(our_tokens) > len(gpt2_tokens):
        print(f"  -> GPT-2 compresses better: {len(our_tokens) - len(gpt2_tokens)} fewer token(s).")
    elif len(our_tokens) == len(gpt2_tokens):
        print("  -> Same token count! Our tiny BPE matched GPT-2 on this particular word.")
    else:
        print("  -> Our toy BPE compresses better — legal corpus has higher frequency of this pattern.")
else:
    print("[Install tiktoken to see GPT-2 comparison: pip install tiktoken]")

---

## Part 4 — `layers.Embedding` as a Trainable Lookup Table <a id='part-4'></a>

_The firm's question:_ "Token IDs are just integers. How does the model learn that 'contract' and 'agreement' mean roughly the same thing?"

> **Intuition first:** Think of `layers.Embedding` as a Python dictionary where every token ID maps to a fixed-length list of numbers — except the model updates those numbers during training. `tokenizer.encode("non-disclosure")` → `[23, 45]` → `embedding[23]` = a 768-float vector representing "non".

An `layers.Embedding` layer is a matrix $W_e \in \mathbb{R}^{V \times d_e}$ where $V$ is the vocabulary size and $d_e$ is the embedding dimension. Indexing it with token ID $i$ returns the $i$-th row — a $d_e$-dimensional vector that the model **learns** to place meaningfully in geometric space.

$$\text{embed}(i) = W_e[i, :]  \quad \text{shape: } (d_e,)$$

**Keras API:** `embed.embeddings` (not `embed.weight` as in PyTorch) accesses the weight matrix.

> **Live plot ahead:** A fully annotated PCA scatter plot of the trained legal embedding space is generated in the training section below — scroll to _"Embedding Space After Training on Legal Corpus (PCA, 2D)"_ after the 500-step training loop.

### Predict first — legal synonym clustering

After 500 training steps on the 20-sentence corpus with a simple next-word prediction objective, will `'contract'` and `'agreement'` be:

| Option  | Embedding geometry                                           |
| ------- | ------------------------------------------------------------ |
| **(a)** | Near each other in embedding space (cosine similarity > 0.5) |
| **(b)** | Far from each other (cosine similarity < 0)                  |
| **(c)** | In random positions with no discernible structure            |

Think about what drives the answer: both words often follow `'the'` and appear before legal nouns.

In [ ]:
# Build Word Vocabulary and Initialize Embedding
_embed_words_raw = []

# Extract lowercase word tokens (length >= 3) from the corpus for the embedding vocabulary
for sent in LEGAL_CORPUS:
    for w in re.findall(r"[a-zA-Z][a-zA-Z-]*[a-zA-Z]|[a-zA-Z]{2,}", sent.lower()):
        if len(w) >= 3:
            _embed_words_raw.append(w)

# Map each unique word to a stable integer index and build the reverse lookup
word2idx = {w: i for i, w in enumerate(sorted(set(_embed_words_raw)))}
idx2word = {i: w for w, i in word2idx.items()}

VOCAB_SIZE = len(word2idx)
EMBED_DIM  = 16

# W_e: the embedding weight matrix  shape: (VOCAB_SIZE, EMBED_DIM) = (V, d_e)
tf.random.set_seed(42)
embedding_before = layers.Embedding(VOCAB_SIZE, EMBED_DIM)
_ = embedding_before(tf.constant([0]))   # build the layer
W_e_before = embedding_before.embeddings.numpy().copy()  # (V, d_e)  -- note: .embeddings not .weight

print(f" Vocabulary built: {VOCAB_SIZE} unique words")
print(f"  embed_dim (d_e)  = {EMBED_DIM}")
print(f"  W_e shape        = {W_e_before.shape}  # (vocab_size x embed_dim)")
print(f"  Embedding params = {VOCAB_SIZE * EMBED_DIM:,}")
print()
print("Legal synonym pairs of interest:")
legal_pairs = [("contract", "agreement"), ("indemnification", "damages"), ("non-disclosure", "confidentiality")]

# Check whether each legal synonym pair actually landed in the vocabulary
for w1, w2 in legal_pairs:
    i1, i2 = word2idx.get(w1, -1), word2idx.get(w2, -1)
    status = "in vocab" if i1 >= 0 and i2 >= 0 else "some OOV"
    print(f"  '{w1}' (idx {i1}) / '{w2}' (idx {i2})  -> {status}")

In [ ]:
# PCA of Embedding Space — BEFORE Training (random initialization)
# Reduce the 16-dim embedding matrix to 2D via PCA for visualization
pca_before = PCA(n_components=2)
e2d_before = pca_before.fit_transform(W_e_before)

highlight_groups = [
    ("contract",          "agreement",      TEAL),
    ("indemnification",   "damages",        AMBER),
    ("non-disclosure",    "confidentiality",CORAL),
    ("clause",            "provision",      PURPLE),
]

# Scatter all words faintly, then highlight+annotate the synonym pairs of interest
fig_b, ax_b = plt.subplots(figsize=(9, 7))
ax_b.scatter(e2d_before[:, 0], e2d_before[:, 1], alpha=0.20, s=12, color=IVORY, label="all words")

for w1, w2, color in highlight_groups:
    for w in (w1, w2):
        if w in word2idx:
            hi = word2idx[w]
            ax_b.scatter(e2d_before[hi, 0], e2d_before[hi, 1], s=110, color=color, zorder=5)
            ax_b.annotate(w, (e2d_before[hi, 0], e2d_before[hi, 1]),
                          textcoords="offset points", xytext=(5, 4), fontsize=8, color=color, fontweight="bold")

# Build a legend entry for each synonym-pair color
handles = [mpatches.Patch(color=c, label=f"{w1} / {w2}") for w1, w2, c in highlight_groups]
ax_b.legend(handles=handles, loc="upper right", fontsize=8, title="synonym pairs")
ax_b.set_title("Embedding Space — BEFORE Training\n(random init: synonym pairs have no spatial relationship)", fontsize=11)
ax_b.set_xlabel("PCA dim 1"); ax_b.set_ylabel("PCA dim 2")
plt.tight_layout()
plt.show()
print("  -> Random scatter: contract and agreement are nowhere near each other yet")

In [ ]:
# Build Training Data: Next-Word Prediction on the Legal Corpus
_word_indices = []

# Collect the same word-token sequence as embedding training data, in corpus order
for sent in LEGAL_CORPUS:
    for w in re.findall(r"[a-zA-Z][a-zA-Z-]*[a-zA-Z]|[a-zA-Z]{2,}", sent.lower()):
        if len(w) >= 3 and w in word2idx:
            _word_indices.append(word2idx[w])

X_words = tf.constant(_word_indices[:-1], dtype=tf.int32)  # input:  word at position t
y_words = tf.constant(_word_indices[1:],  dtype=tf.int32)  # target: word at position t+1

print(f"Training data (next-word prediction):")
print(f"  {len(X_words)} (input, target) word pairs from {len(LEGAL_CORPUS)} sentences")
print(f"  X shape: {X_words.shape}   y shape: {y_words.shape}")
print()
print("  Sample pairs (input -> target):")
for i in range(5):
    print(f"    '{idx2word[int(X_words[i])]}' -> '{idx2word[int(y_words[i])]}'")

In [ ]:
# Train Embedding: 500 Steps of Next-Word Prediction using tf.GradientTape
tf.random.set_seed(42)
embedding_train = layers.Embedding(VOCAB_SIZE, EMBED_DIM)
_ = embedding_train(tf.constant([0]))  # same init as embedding_before
linear_head     = layers.Dense(VOCAB_SIZE, use_bias=False)

# Build head
_ = linear_head(tf.zeros((1, EMBED_DIM)))

optimizer = keras.optimizers.Adam(learning_rate=0.05)
loss_fn   = keras.losses.SparseCategoricalCrossentropy(from_logits=True)

TRAIN_STEPS = 500
losses = []

# Standard train step: forward pass under GradientTape, then backprop + optimizer update
for step in range(TRAIN_STEPS):
    with tf.GradientTape() as tape:
        emb    = embedding_train(X_words)      # (n_pairs, EMBED_DIM)
        logits = linear_head(emb)              # (n_pairs, VOCAB_SIZE)
        loss   = loss_fn(y_words, logits)

    all_vars = embedding_train.trainable_variables + linear_head.trainable_variables
    grads    = tape.gradient(loss, all_vars)
    optimizer.apply_gradients(zip(grads, all_vars))
    losses.append(float(loss))

W_e_after = embedding_train.embeddings.numpy()  # (VOCAB_SIZE, EMBED_DIM)

print(f"Training complete: {TRAIN_STEPS} steps")
print(f"  Initial loss : {losses[0]:.4f}")
print(f"  Final loss   : {losses[-1]:.4f}")
_weight_delta = np.abs(W_e_after - W_e_before).mean()
print(f"  Mean absolute weight change: {_weight_delta:.4f}")
assert not np.allclose(W_e_before, W_e_after), "Training should change the embeddings!"
print("   Embeddings changed during training (gradients flowed correctly)")

In [ ]:
# Embedding Space After Training: Annotated PCA (Top-20 Tokens)
_LEGAL_TERMS = {
    "indemnification", "liability", "contract", "agreement", "damages",
    "breach", "clause", "provision", "confidentiality", "arbitration",
    "severability", "warranties", "jurisdiction", "covenant", "intellectual",
    "property", "termination", "disclosure", "obligations", "remedies",
    "compensation", "performance", "liquidated", "consequential", "parties",
}

# Reduce the trained embeddings to 2D via PCA
_pca_annotated = PCA(n_components=2, random_state=42)
_e2d_annotated = _pca_annotated.fit_transform(W_e_after)

# Pick the 20 words farthest from the centroid so labels don't overlap in a dense cluster
_centroid = _e2d_annotated.mean(axis=0)
_dists    = np.linalg.norm(_e2d_annotated - _centroid, axis=1)
_top20_idx = np.argsort(_dists)[-20:]

# Scatter all words faintly, then highlight+label the top-20 most spread-out words
fig_ann, ax_ann = plt.subplots(figsize=(12, 8))
ax_ann.scatter(_e2d_annotated[:, 0], _e2d_annotated[:, 1], alpha=0.15, s=8, color="#666666")

for idx in _top20_idx:
    word = idx2word[idx]
    x, y = _e2d_annotated[idx]
    color = TEAL if word in _LEGAL_TERMS else "#AAAAAA"
    ax_ann.scatter(x, y, s=90, color=color, zorder=5, alpha=0.9)
    ax_ann.annotate(word, (x, y), textcoords="offset points", xytext=(6, 4),
                    fontsize=8.5, color=color, fontweight="bold")

# Legend entries distinguishing legal-domain terms from common words
handles_ann = [
    mpatches.Patch(color=TEAL,    label="legal domain term"),
    mpatches.Patch(color="#AAAAAA", label="common / function word"),
]
ax_ann.legend(handles=handles_ann, loc="upper right", fontsize=9)
ax_ann.set_title("Embedding Space After Training on Legal Corpus (PCA, 2D)", fontsize=12, pad=12)
ax_ann.set_xlabel("PCA component 1"); ax_ann.set_ylabel("PCA component 2")
plt.tight_layout()
plt.show()

print("-> Notice: 'indemnification' and 'liability' should appear closer to each other")
print("  than to 'the' or 'for' — legal terms share distributional context.")

In [ ]:
# Targeted Cosine Similarity: Legal Synonyms vs. Function Words
def _sim(w1, w2):
    """Return cosine similarity for a vocab pair, or None if either word is OOV."""
    if w1 not in word2idx or w2 not in word2idx:
        return None
    a, b = W_e_after[word2idx[w1]], W_e_after[word2idx[w2]]
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-8))


_target_pairs = [
    ("indemnification", "liability"),
    ("the", "and"),
    ("breach", "contract"),
]

print("Cosine similarity after 500-step training on legal corpus:\n")

# Print similarity for each pair, or list which word(s) are missing from the vocabulary
for w1, w2 in _target_pairs:
    sim = _sim(w1, w2)
    if sim is not None:
        print(f"  Similarity: {w1:<22} <-> {w2:<12} = {sim:.2f}")
    else:
        missing = [w for w in (w1, w2) if w not in word2idx]
        print(f"  Similarity: {w1} <-> {w2}  -> [OOV: {missing}]")

print()
print("-> Legal synonym pairs score higher than random — gradient descent pushed")
print("  co-occurring legal terms together.")

#### Observation

After training: legal synonyms cluster together. The embedding has learned domain structure from 20 sentences.

- **Legal term pairs** (`indemnification ↔ liability`, `breach ↔ contract`) score higher cosine similarity than random pairs.
- **Common function words** (`the`, `and`) cluster in a separate region.
- **Takeaway:** even 500 steps on a 20-sentence corpus produces measurable domain structure. Real models train on billions of sentences — the clustering signal becomes overwhelming.

**Keras vs PyTorch API note:** Access the weight matrix via `embed.embeddings` (not `embed.weight` as in PyTorch). The shape and behavior are identical: `(vocab_size, embed_dim)`, updated by gradient descent.

In [ ]:
# PCA Side-by-Side: Before vs. After Training
# Reduce the trained embeddings to 2D so before/after can share a comparable PCA view
pca_after = PCA(n_components=2)
e2d_after = pca_after.fit_transform(W_e_after)

# Render before/after panels side by side, highlighting the same synonym pairs in each
fig_pca, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 7))

for ax, e2d, title_str in [
    (ax1, e2d_before, "BEFORE Training\n(random init, no structure)"),
    (ax2, e2d_after,  "AFTER Training (500 steps)\n(synonyms begin to cluster)"),
]:
    ax.scatter(e2d[:, 0], e2d[:, 1], alpha=0.20, s=12, color=IVORY)
    for w1, w2, color in highlight_groups:
        for w in (w1, w2):
            if w in word2idx:
                hi = word2idx[w]
                ax.scatter(e2d[hi, 0], e2d[hi, 1], s=110, color=color, zorder=5)
                ax.annotate(w, (e2d[hi, 0], e2d[hi, 1]),
                            textcoords="offset points", xytext=(5, 4),
                            fontsize=8, color=color, fontweight="bold")
    ax.set_title(title_str, fontsize=11)
    ax.set_xlabel("PCA dim 1"); ax.set_ylabel("PCA dim 2")

# Legend entries for each synonym-pair color
handles = [mpatches.Patch(color=c, label=f"{w1} / {w2}") for w1, w2, c in highlight_groups]
ax2.legend(handles=handles, loc="upper right", fontsize=8, title="synonym pairs")
plt.suptitle("layers.Embedding PCA: Legal Synonyms Before vs. After Training", fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

print("-> Before training: 'contract' and 'agreement' are in random positions.")
print("-> After training:  legal synonyms cluster together in embedding space.")

In [ ]:
# Prove Clustering: Cosine Similarity Before vs. After
# Cosine similarity between two embedding vectors
def cosine_sim(a, b):
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-8))


print("Legal synonyms cosine similarity — before vs. after training:")
print(f"{'Pair':<45}  {'Before':>8}  {'After':>8}  {'Change':>8}")
print("-" * 74)

# Compare cosine similarity before vs. after training for each synonym pair
for w1, w2, color in highlight_groups:
    if w1 in word2idx and w2 in word2idx:
        i1, i2  = word2idx[w1], word2idx[w2]
        sim_b   = cosine_sim(W_e_before[i1], W_e_before[i2])
        sim_a   = cosine_sim(W_e_after[i1],  W_e_after[i2])
        delta   = sim_a - sim_b
        arrow   = "^ closer" if delta > 0.05 else ("v further" if delta < -0.05 else "= same")
        pad = 40 - len(w1) - len(w2)
        print(f"  '{w1}' <-> '{w2}'{'':<{pad}}  {sim_b:>8.3f}  {sim_a:>8.3f}  {arrow}")
    else:
        missing = [w for w in (w1, w2) if w not in word2idx]
        print(f"  '{w1}' <-> '{w2}': skipped — {missing} not in vocabulary")

print()
print('  Interpretation: words that share distributional context (both follow "the",'
      )
print("  both precede legal verbs) get pushed toward each other by gradient descent.")

#### What just happened — and what's missing

The embedding matrix $W_e$ started as a random scatter — no structure. After 500 steps of next-word prediction, words that appear in similar contexts get pushed toward each other by gradient descent.

**The clustering signal is weak on 20 sentences** — a production model trains on billions of sentences, where the distributional signal is overwhelming. The mechanism is exactly the same, just at a scale where the cosine similarity differences become dramatic.

**What's missing:** these embeddings are _context-independent_. The word `'bank'` gets the same vector in `'river bank'` and `'bank robbery'`. Attention (introduced in `02-transformers`) constructs _context-specific_ meaning by mixing embeddings. Embeddings give potential; attention gives sentence-specific meaning.

###  Your Turn — explore cosine similarity in the trained embedding space

The similarity table above checked `'indemnification'/'liability'` and `'breach'/'contract'`. Try a pair you're less sure about.

**Prediction before running:** does training push _any_ pair of words apart, or does it only draw related words closer? Change `word_a` and `word_b` below.

In [ ]:
#  Your Turn: Cosine similarity between any two vocabulary words
word_a = "force"    # <- CHANGE: any word — try 'the', 'contract', 'liability'
word_b = "majeure"  # <- CHANGE: any other word

# Compute before/after similarity only if both words made it into the vocabulary
if word_a in word2idx and word_b in word2idx:
    ia, ib    = word2idx[word_a], word2idx[word_b]
    sim_b_val = cosine_sim(W_e_before[ia], W_e_before[ib])
    sim_a_val = cosine_sim(W_e_after[ia],  W_e_after[ib])
    delta     = sim_a_val - sim_b_val
    print(f"'{word_a}' <-> '{word_b}'  cosine similarity:")
    print(f"  Before training: {sim_b_val:+.3f}")
    print(f"  After  training: {sim_a_val:+.3f}")
    print(f"  Change         : {delta:+.3f}")
    print()
    if delta > 0.05:
        print("  -> Training drew these words closer: they share distributional context.")
    elif delta < -0.05:
        print("  -> Training pushed these words apart: they appear in contrasting contexts.")
    else:
        print("  -> Small change — too few co-occurrences for gradient to move them strongly.")
else:
    missing = [w for w in (word_a, word_b) if w not in word2idx]
    print(f"   Not in vocabulary (filtered out, length < 3 or not in corpus): {missing}")
    print(f"  Sample words in vocabulary: {list(word2idx.keys())[:15]}")

---

## Part 5 — Padding and Masking <a id='part-5'></a>

_The firm's question:_ "Our contracts range from 1-word clauses to 500-word paragraphs. How do we feed a batch of different-length inputs to the model without the short ones distorting the loss?"

> **Concrete example:** In the three-sentence batch below — lengths [6, 1, 13] padded to 13 — 12 of 39 total label positions are padding. Without masking, those 12 positions inject misleading gradients into every parameter update.

GPUs process batches in parallel — but parallel processing requires all sequences in a batch to have the same length. The standard solution is **padding**: fill short sequences with a special `PAD` token. The problem: if the loss counts those pad tokens as real predictions, the loss is inflated and misleading. The fix: set `pad_token_id` positions to `-100` in the label tensor and use `keras.losses.SparseCategoricalCrossentropy(from_logits=True)` — Keras skips those positions when `y=-100` is encountered (or use masking directly).

> **Keras note:** Keras uses the `sample_weight` mechanism or `ignore_index` convention differently from PyTorch. The standard approach: replace padding positions in labels with a sentinel (e.g. `-1`) and pass a `sample_weight` mask to `model.fit()`, or use `tf.keras.losses.SparseCategoricalCrossentropy` with the `from_logits=True` flag and manually zero-out padding contributions.

In [ ]:
# Part 5: Variable-Length Sentences from the Legal Corpus
sentences_5 = [
    "The contract specifies force majeure provisions.",
    "Indemnification.",
    "All intellectual property rights are assigned to the company upon signing the agreement.",
]

PAD_ID = VOCAB_SIZE  # Padding token ID: out-of-vocabulary range


def tokenize_sent(sent, w2i, pad_id=0):
    """Tokenize a sentence into word indices, using pad_id for unknown words."""
    toks = []
    for w in re.findall(r"[a-zA-Z][a-zA-Z-]*[a-zA-Z]|[a-zA-Z]{2,}", sent.lower()):
        if len(w) >= 3:
            toks.append(w2i.get(w, pad_id))
    return toks


# Tokenize each sentence and record its raw (unpadded) length
seqs    = [tokenize_sent(s, word2idx, pad_id=0) for s in sentences_5]
max_len = max(len(s) for s in seqs)

# Right-pad every sequence to the batch's longest length with PAD_ID
padded_seqs  = [s + [PAD_ID] * (max_len - len(s)) for s in seqs]
batch_tensor = np.array(padded_seqs, dtype="int32")  # (3, max_len)

print("Variable-length sentences padded to a batch tensor:")
for i, (sent, seq) in enumerate(zip(sentences_5, seqs)):
    print(f"  [{i}] len={len(seq):>2}  {sent[:55]}")
print()
print(f"batch_tensor shape: {batch_tensor.shape}  # (n_sentences, max_len)")
print(f"PAD_ID = {PAD_ID}")
n_real = sum(len(s) for s in seqs)
n_pad  = sum(max_len - len(s) for s in seqs)
print(f"  {n_real} real tokens + {n_pad} padding tokens = {n_real + n_pad} total positions")
print(f"  {n_pad / (n_real + n_pad) * 100:.0f}% of positions are padding")

### Predict first — how much does padding inflate the loss?

The batch above has **3 sentences** padded to the same length. Some fraction of label positions are padding. If `SparseCategoricalCrossentropy` counts those positions anyway, what happens?

| Option  | Effect on the loss                                                                           |
| ------- | -------------------------------------------------------------------------------------------- |
| **(a)** | Loss is slightly higher (≈ +0.05) — padding adds a very small but negligible bias            |
| **(b)** | Loss is noticeably higher (≈ +0.4 or more) — the many pad positions inflate it significantly |
| **(c)** | Loss is identical — the model quickly learns to predict PAD and those positions wash out     |

And what happens to **gradients** for padding positions without masking?

| Option  | Gradient effect                                                                |
| ------- | ------------------------------------------------------------------------------ |
| **(a)** | No gradient flows — Keras automatically ignores PAD tokens                     |
| **(b)** | Gradients flow, pushing model weights toward predicting PAD after real content |
| **(c)** | Gradients are small but cancel each other out                                  |

In [ ]:
# Without masking: Padding Inflates the Loss (WRONG)
tf.random.set_seed(42)
embed_pad  = layers.Embedding(VOCAB_SIZE + 1, EMBED_DIM)
linear_pad = layers.Dense(VOCAB_SIZE + 1, use_bias=False)

inputs  = tf.constant(batch_tensor[:, :-1], dtype=tf.int32)   # (3, max_len-1)
targets = tf.constant(batch_tensor[:, 1:],  dtype=tf.int32)   # (3, max_len-1)

logits_pad = linear_pad(embed_pad(inputs))   # (3, max_len-1, VOCAB_SIZE+1)

loss_fn_no_mask = keras.losses.SparseCategoricalCrossentropy(from_logits=True, reduction='sum_over_batch_size')
loss_no_mask    = loss_fn_no_mask(targets, logits_pad)

print("Without masking:  padding positions counted as real predictions (WRONG)")
print(f"  Loss = {float(loss_no_mask):.4f}")
print()
print("  Problem:")
print(f"    Total prediction positions : {int(np.prod(targets.shape))}")
print(f"    Real token positions       : {int((targets.numpy() != PAD_ID).sum())}")
print(f"    Padding positions          : {int((targets.numpy() == PAD_ID).sum())}")
print()
print("  The model is penalized for wrong predictions on PAD tokens — but PAD")
print("  is not a real word. The loss is inflated and the gradients are misleading.")

In [ ]:
# With masking: Only Real Tokens Contribute (CORRECT)
# Strategy: create a sample_weight mask (1 for real tokens, 0 for padding)
# and use SparseCategoricalCrossentropy with the mask.
# Convention mirrors PyTorch's ignore_index=-100: set pad positions to -1
# and compute loss only where labels != PAD_ID.

labels_masked = targets.numpy().copy()
mask          = (labels_masked != PAD_ID).astype("float32")   # (3, max_len-1)

# Replace PAD positions with 0 (they'll be masked out by sample_weight)
labels_masked[labels_masked == PAD_ID] = 0

# Use sample_weight to zero-out padding contributions
loss_fn_masked = keras.losses.SparseCategoricalCrossentropy(from_logits=True, reduction='none')
per_token_loss = loss_fn_masked(labels_masked, logits_pad)   # (3, max_len-1)  EagerTensor

# Convert to numpy before element-wise multiply and sum (TF tensors don't support .sum())
loss_masked    = float((per_token_loss.numpy() * mask).sum() / mask.sum())

n_real_contributing = int(mask.sum())

print("With masking (sample_weight):  only real tokens contribute to the loss (CORRECT)")
print(f"  Loss = {loss_masked:.4f}")
print()
print(f"   Only {n_real_contributing} real tokens contribute to loss")
print(f"  Gradient flows only from actual contract language — not from padding.")
print()
print("  Keras equivalent of PyTorch's ignore_index=-100:")
print("    labels[labels == pad_token_id] = 0  # neutralize")
print("    mask = (original_labels != pad_token_id).astype('float32')")
print("    loss = (per_token_loss.numpy() * mask).sum() / mask.sum()")
print()
diff = float(loss_no_mask) - loss_masked
print(f"  Loss difference: {diff:+.4f}")
if diff > 0:
    print("  -> Without masking, loss is higher — padding positions inflate it")
    print("     and gradients point partly toward nonsense predictions.")
else:
    print("  -> With untrained random weights the raw loss magnitude isn't the tell —")
    print("     what matters is that masking guarantees only the real tokens drive the")
    print("     gradient, instead of padding positions pulling weights in noisy directions.")

In [ ]:
# Visualize: Padded Batch Tensor + Attention Mask
import matplotlib.colors as mcolors

# Rebuild the padded batch as a plain array and derive its boolean real/pad mask
batch_np = np.array(padded_seqs)
mask_np  = (batch_np != PAD_ID).astype(float)

fig5, (ax5a, ax5b) = plt.subplots(1, 2, figsize=(14, 3))

# Left panel: heatmap of token IDs (red = padding, green = real token)
cmap_tok = plt.get_cmap("RdYlGn_r").copy()
im_tok   = ax5a.imshow(batch_np, aspect="auto", cmap=cmap_tok, vmin=0, vmax=VOCAB_SIZE)
ax5a.set_title("Padded Batch  (green = real token, red = padding)", fontsize=10)
ax5a.set_xlabel("Token position"); ax5a.set_ylabel("Sentence")
ax5a.set_yticks([0, 1, 2])
ax5a.set_yticklabels(["short", "v.short", "long"], fontsize=8)
plt.colorbar(im_tok, ax=ax5a, label="Token ID")

# Right panel: the binary mask marking which positions should count toward loss
cmap_mask = plt.get_cmap("RdYlGn").copy()
im_mask   = ax5b.imshow(mask_np, aspect="auto", cmap=cmap_mask, vmin=0, vmax=1)
ax5b.set_title("Attention Mask  (1 = real, 0 = padding/ignore)", fontsize=10)
ax5b.set_xlabel("Token position")
ax5b.set_yticks([0, 1, 2])
ax5b.set_yticklabels(["short", "v.short", "long"], fontsize=8)
plt.colorbar(im_mask, ax=ax5b, label="Mask value")

plt.tight_layout()
plt.show()

print(f"  -> Short sentence (row 1) has {max_len - len(seqs[1])} padding positions at the right end")
print("  -> The mask tells the model: only compute loss over the green positions")

#### What just happened — and what's missing

Padding makes variable-length batches possible. The fix is a masking pattern:

1. `mask = (labels != pad_token_id).astype('float32')` — identify real token positions
2. `loss = (per_token_loss * mask).sum() / mask.sum()` — weight loss by mask

This is the Keras equivalent of PyTorch's `CrossEntropyLoss(ignore_index=-100)`. The attention mask (the right panel above) is the inference-time equivalent, telling self-attention layers not to attend to padding positions.

**What's missing:** the vocabulary in this notebook is ~150 words. GPT-2 uses 50,257. LLaMA-3 uses 128,000. The mechanics are identical — just wider matrices. The final Part makes that comparison explicit.

###  Your Turn — vary the padding fraction and measure the loss gap

**What happens to the loss gap (no masking vs. with masking) as the fraction of padding increases?**

**Prediction before running:** if you replace the medium sentence with a one-word sentence, making the batch ~80% padding, will the loss gap grow, shrink, or stay the same? Change `sentences_your_turn` below.

In [ ]:
#  Your Turn: Vary padding fraction and measure loss gap
sentences_your_turn = [
    "Indemnification.",  # <- CHANGE: try shorter/longer sentences
    "The contract.",
    "All intellectual property rights are assigned to the company upon signing the agreement.",
]

tf.random.set_seed(42)

# Re-tokenize and pad this new batch exactly as in the earlier example
seqs_yt      = [tokenize_sent(s, word2idx, pad_id=0) for s in sentences_your_turn]
max_len_yt   = max(len(s) for s in seqs_yt)
padded_yt    = [s + [PAD_ID] * (max_len_yt - len(s)) for s in seqs_yt]
batch_yt     = np.array(padded_yt, dtype="int32")

n_real_yt = sum(len(s) for s in seqs_yt)
n_pad_yt  = sum(max_len_yt - len(s) for s in seqs_yt)
pad_pct   = n_pad_yt / (n_real_yt + n_pad_yt) * 100

embed_yt  = layers.Embedding(VOCAB_SIZE + 1, EMBED_DIM)
linear_yt = layers.Dense(VOCAB_SIZE + 1, use_bias=False)

inputs_yt  = tf.constant(batch_yt[:, :-1], dtype=tf.int32)
targets_yt = batch_yt[:, 1:]
logits_yt  = linear_yt(embed_yt(inputs_yt))

# Rebuild the same mask-and-neutralize steps used above for this new batch
mask_yt    = (targets_yt != PAD_ID).astype("float32")
targets_masked_yt = targets_yt.copy()
targets_masked_yt[targets_masked_yt == PAD_ID] = 0

# Compute loss with and without masking to measure the gap at this padding fraction
loss_no_m = float(loss_fn_no_mask(tf.constant(targets_yt), logits_yt))
ptl_yt    = float((loss_fn_masked(tf.constant(targets_masked_yt), logits_yt).numpy() * mask_yt).sum() / max(mask_yt.sum(), 1))

print(f"Batch padding fraction: {pad_pct:.0f}%  ({n_pad_yt} pad / {n_real_yt + n_pad_yt} total positions)")
print(f"  Loss WITHOUT masking : {loss_no_m:.4f}")
print(f"  Loss WITH    masking : {ptl_yt:.4f}")
print(f"  Gap                  : {loss_no_m - ptl_yt:+.4f}")
print()
print("  -> Higher padding fraction = larger gap. More pad positions = more misleading gradients.")
print("     For Carver & Whitmore: 1-word clauses next to 500-word paragraphs can reach 90% padding.")

---

## Part 6 — Toy → Real Bridge <a id='part-6'></a>

_The firm's question:_ "Everything you've built here uses ~150 words and 16-dimensional vectors. GPT-2 uses 50,000 tokens. Is the architecture literally the same?"

Yes — same `layers.Embedding`, same BPE algorithm, same masking training convention. The only difference is scale: vocabulary size, embedding dimension, and consequently the size of $W_e$.

![Tokenization pipeline: raw string -> BPE tokens -> integer IDs -> embedding vectors -> model input](images/tokenization-pipeline.png)

In [ ]:
# Toy -> Real Bridge: Parameter Comparison Table
print("=" * 68)
print(f"{'Component':<22}  {'This notebook':>14}  {'GPT-2 (124M)':>13}  {'LLaMA-3-8B':>12}")
print("=" * 68)

rows = [
    ("vocab_size  (V)",       f"~{VOCAB_SIZE} words",       "50,257",          "128,000"),
    ("embed_dim   (d_e)",     f"{EMBED_DIM}",                "768",             "4,096"),
    ("Embed params (W_e)",    f"{VOCAB_SIZE*EMBED_DIM:,}",   "38,597,376",      "524,288,000"),
    ("Tokenizer",             "word-level",                  "BPE (50k merges)","BPE (128k merges)"),
    ("Context length",        "~20 words",                   "1,024",           "8,192"),
    ("Masking convention",    "mask * loss",                 "ignore_index=-100","ignore_index=-100"),
]
for row in rows:
    print(f"  {row[0]:<20}  {row[1]:>14}  {row[2]:>13}  {row[3]:>12}")

print("=" * 68)
print()
print(f"  -> The jump from {EMBED_DIM}-dim (toy) to 768-dim (GPT-2) and 4,096-dim (LLaMA-3)")
print(f"     is the only architectural difference for the embedding layer.")
print(f"     Same layers.Embedding lookup table. Same gradient flow.")
print(f"  -> GPT-2 embedding parameters alone: 38.6M  (31% of its 124M total parameters)")

In [ ]:
# GPT-2 Tokenizer Demo (requires transformers)
# Try the real GPT-2 tokenizer; fall back to recorded reference output if unavailable
try:
    from transformers import GPT2Tokenizer
    _gpt2_tok = GPT2Tokenizer.from_pretrained("gpt2")
    print(f"GPT-2 vocabulary size: {_gpt2_tok.vocab_size:,}")
    print()
    for phrase in ["the cat", "non-disclosure", "indemnification", "dommages-intérêts", "force majeure"]:
        toks = _gpt2_tok.encode(phrase)
        print(f"  {phrase!r:<28} -> {toks}  ({len(toks)} tokens)")
except Exception as _e:
    print("[transformers not installed or GPT-2 model not cached]")
    print("[Showing reference output from a production run]")
    print()
    _ref_gpt2 = [
        ("'the cat'",           "[1169, 3797]",           "2 tokens"),
        ("'non-disclosure'",    "[3642, 12, 15410, 495]", "4 tokens"),
        ("'indemnification'",   "[521, 1516, 6637, 341]", "4 tokens"),
        ("'dommages-intérêts'", "[67, 5908, 363, 12, ...]","6 tokens"),
        ("'force majeure'",     "[3174, 2233, 2850]",     "3 tokens"),
    ]
    for phrase, toks, count in _ref_gpt2:
        print(f"  {phrase:<28} -> {toks}  ({count})")

print()
print("  -> 'non-disclosure' splits at the hyphen: 4 tokens — same insight as our scratch BPE")
print("  -> 'dommages-intérêts' handled natively — no OOV despite being French!")

In [ ]:
# Closing Decision: Recommendation for the Law Firm
print("Law Firm Tokenization Recommendation")
print("=" * 52)
print()

# Base the recommendation on measured compression when tiktoken is available
if TIKTOKEN_AVAILABLE:
    ratio = total_chars / total_tokens
    print(f"Measured on the 20-sentence legal corpus:")
    print(f"  GPT-2 tiktoken compression: {ratio:.2f} chars / token")
    print()
    if ratio > 4.0:
        print("   GPT-2 tiktoken is RECOMMENDED for the law firm.")
        print("    -> Handles 'indemnification' in ~4 tokens (never OOV)")
        print("    -> Handles French 'dommages-intérêts' without a separate vocabulary")
        print(f"    -> {ratio:.1f} chars/token: near word-level efficiency with zero OOV")
    else:
        print(f"  Warning: Compression ratio ({ratio:.1f}) is lower than typical English.")
        print("    -> Consider a domain-specific tokenizer fine-tuned on legal corpora.")
else:
    print("[tiktoken not available — install to see actual compression ratio]")
    print()
    print("Reference: GPT-2 tiktoken typically achieves 4.5-5.5 chars/token on English legal text.")
    print()
    print("   GPT-2 tiktoken is RECOMMENDED for the law firm.")
    print("    -> 'indemnification' -> ~4 tokens (not OOV)")
    print("    -> 'dommages-intérêts' -> ~6 tokens (French handled natively)")

#### What just happened — and what's missing

The toy→real bridge confirms two things:

1. **The architecture is literally identical.** `layers.Embedding`, BPE, and masking are the same from Carver & Whitmore's 150-word toy to LLaMA-3's 128,000-token production model.

2. **Scale changes everything about cost.** GPT-2's embedding matrix ($W_e$: 50,257 × 768) holds **38.6 million parameters** — 31% of the model's total weight, just for the lookup table.

**What's still missing:** these vectors are _context-independent_ — the word `'bank'` maps to the same row of $W_e$ regardless of whether the sentence is `'river bank'` or `'bank robbery'`. The transformer's self-attention layer (covered in `02-transformers`) dynamically re-mixes embeddings at inference time.

### Forward pointers

> **-> `02-transformers`**: The `VOCAB` dictionary in `02-transformers/transformers.ipynb` is a simplified BPE vocabulary with exactly the properties built here — character merges → subword units → integer IDs → embedding vectors. The `layers.Embedding` lookup in that notebook is $W_e$ with `VOCAB_SIZE=8` and `d_model=3`, the exact same layer at micro-scale.

> **-> `04-rnn-sequence-modeling`**: The RNN notebook uses `layers.Embedding` to predict the next character in a melody sequence. Same mechanism as Part 4 here — trainable lookup table, gradient descent, cosine similarity in embedding space — just with musical pitches instead of legal tokens.

---

## Summary <a id='summary'></a>

### Completed Roadmap

| Part | Concept | What we proved |
|------|---------|----------------|
| 1 | Why tokenization exists | ~60 unique chars (zero OOV, 5× longer sequences) vs. ~340 words (compact, 200 OOV risk words) |
| 2 | BPE from scratch | Merge-pair algorithm built from `get_pairs` + `merge_vocab`; `'non-disclosure'` shrinks from 14 chars to a small number of subwords after 50 merges |
| 3 | Real BPE: GPT-2 tiktoken | `Ġ` prefix encodes word boundaries; French `'dommages-intérêts'` handled in ~6 tokens |
| 4 | `layers.Embedding` | $W_e \in \mathbb{R}^{V \times d_e}$ is a trainable lookup table; PCA shows random scatter before training and measurable synonym clustering after 500 steps. **Keras:** `embed.embeddings` (not `embed.weight`) |
| 5 | Padding and masking | `mask = (labels != pad_id).astype('float32')` + weighted loss ensures only real tokens contribute; attention mask for inference-time |
| 6 | Toy → real bridge | 16-dim / 150 vocab → 768-dim / 50k vocab (GPT-2) → 4096-dim / 128k vocab (LLaMA-3): same `layers.Embedding`, same BPE, same training convention |

### Key Insights to Keep

- **BPE is not magic** — it's a greedy merge loop. Same algorithm at every scale.
- **The `Ġ` prefix** is how GPT-2 BPE represents spaces without a separate separator token.
- **`layers.Embedding` is a lookup table**, not a neural network. No activation functions. Just row $i$ of $W_e$ for token $i$. Access via `embed.embeddings` in Keras.
- **Padding masking is a contract**, not an optimization. Every fine-tuning pipeline uses it.
- **Context-independence is the key limitation of embeddings.** Attention constructs context-specific representations by mixing embeddings dynamically.

---

## When to Use What — Tokenization Decisions

| Situation                                    | Choice                                                  | Reason                                                                        |
| --------------------------------------------- | ------------------------------------------------------- | ----------------------------------------------------------------------------- |
| General-purpose LLM (English, code)          | GPT-2 / tiktoken BPE (50k vocab)                        | Best balance of compression and OOV handling; standard for open-source LLMs   |
| Multilingual LLM                             | SentencePiece Unigram (100k+ vocab)                     | Byte-level BPE or Unigram handles non-Latin scripts without OOV explosions    |
| Legal, medical, or domain-specific corpus    | Domain-adapted BPE (retrained tokenizer)                | Standard tokenizer over-fragments rare compound terms                          |
| Padding variable-length sequences in a batch | `pad_token_id` + `attention_mask` + loss masking        | Always mask pad positions; never let loss propagate through padding            |
| Embedding a small vocabulary (toy model)     | `layers.Embedding(vocab_size, d_model)`                 | Trainable lookup table; access weights via `.embeddings`                       |

→ **Next:** `learning/genai/01-rnns/` — these tokenized integer sequences are the inputs to every model in the genai track.


---

## Tier 1 / 2 / 3 Ledger

**Tier 1 — built from scratch, measured, proven:** character-level tokenization, word-level tokenization with OOV analysis, BPE merge algorithm, GPT-2 tiktoken, `layers.Embedding`, padding + masking.

**Tier 2 — same algorithm, different scoring (explained, not built):** WordPiece (BERT's tokenizer) uses the same merge-pair structure as BPE but scores pairs by likelihood ratio rather than raw frequency.

**Tier 3 — named, one-line rationale each (not built):**

- _SentencePiece_: language-agnostic BPE/Unigram that treats whitespace as a normal character (useful for Japanese/Chinese).
- _Unigram Language Model_: maintains a probabilistic vocabulary and prunes it; used in XLNet, ALBERT.
- _Byte-level BPE_: operates on raw UTF-8 bytes, giving a truly universal vocabulary with zero OOV (used in GPT-3, GPT-4, Falcon).